In [1]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import copy
import time
torch.autograd.set_detect_anomaly(True)

mnist_trainset = datasets.MNIST(root='./data/mnist', train=True, download=True, transform=transforms.ToTensor())
mnist_fashion_trainset = datasets.FashionMNIST(root='./data/fashion_mnist', train=True, download=True, transform=transforms.ToTensor())
cifar_trainset = datasets.CIFAR10(root='./data/cifar10', train=True, download=True, transform=transforms.ToTensor())
# Getting mnist test data
mnist_testset = datasets.MNIST(root='./data/mnist', train=False, download=True, transform=transforms.ToTensor())
mnist_fashion_testset = datasets.FashionMNIST(root='./data/fashion_mnist', train=False, download=True, transform=transforms.ToTensor())
cifar_testset = datasets.CIFAR10(root='./data/cifar10', train=False, download=True, transform=transforms.ToTensor())

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(28*28, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))  # Use non-in-place activation (torch.relu instead of relu_)
        x = self.fc2(x)
        return x

# Load data
train_X = mnist_trainset.data
train_y = mnist_trainset.targets
train_X = train_X / 255.0  # Normalize the data
train_X = train_X.to(device).float()
train_y = torch.tensor(train_y, dtype=torch.int64).to(device)

dataloader = torch.utils.data.DataLoader(list(zip(train_X, train_y)), batch_size=64, shuffle=True)

# Model, optimizer, loss function
model_SGD = Net().to(device)
optimizer_SGD = torch.optim.SGD(model_SGD.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
epochs = 5

# Training loop
for epoch in range(epochs):
    model_SGD.train()
    for i, (x, y) in enumerate(dataloader):
        optimizer_SGD.zero_grad()
        y_pred_SGD = model_SGD(x)
        loss_SGD = criterion(y_pred_SGD, y)
        
        # Ensure no in-place operation here
        loss_SGD.backward(create_graph=True)  # Do not modify the loss or gradients in place
        print(f"Epoch {epoch+1}/{epochs}, Batch {i+1}/{len(dataloader)}, Loss: {loss_SGD.item()}")
        optimizer_SGD.step()

# Calculate second-order derivatives (Hessian)
nodewise_traces = []

for param in model_SGD.parameters():
    if param.grad is not None:
        param_trace = 0
        for grad_element in param.grad.view(-1):
            second_order_derivative = torch.autograd.grad(grad_element, param, retain_graph=True, create_graph=True)[0]
            param_trace += second_order_derivative.sum()
        nodewise_traces.append(param_trace)

print("Nodewise Traces of the Hessian:")
print(nodewise_traces)


Files already downloaded and verified
Files already downloaded and verified


C:\Users\Rajeev Wankar\AppData\Local\Temp\ipykernel_21488\2788950571.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_y = torch.tensor(train_y, dtype=torch.int64).to(device)
d:\miniconda3\envs\deephessian\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Using backward() with create_graph=True will create a reference cycle between the parameter and its gradient which can cause a memory leak. We recommend using autograd.grad when creating the graph to avoid this. If you have to use this function, make sure to reset the .grad fields of your parameters to None after use to break the cycle and avoid the leak. (Triggered internally at C:\b\abs_fakvb73nko\croot\pytorch-select_1730848725921\work\torch\csrc\autograd\engine.cpp:1208.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the ba

Epoch 1/5, Batch 1/938, Loss: 2.3019824028015137
Epoch 1/5, Batch 2/938, Loss: 2.297055244445801
Epoch 1/5, Batch 3/938, Loss: 2.3109936714172363
Epoch 1/5, Batch 4/938, Loss: 2.295905351638794
Epoch 1/5, Batch 5/938, Loss: 2.3333535194396973
Epoch 1/5, Batch 6/938, Loss: 2.3061580657958984
Epoch 1/5, Batch 7/938, Loss: 2.299499273300171
Epoch 1/5, Batch 8/938, Loss: 2.3157191276550293
Epoch 1/5, Batch 9/938, Loss: 2.288154125213623
Epoch 1/5, Batch 10/938, Loss: 2.3113014698028564
Epoch 1/5, Batch 11/938, Loss: 2.285098075866699
Epoch 1/5, Batch 12/938, Loss: 2.328505039215088
Epoch 1/5, Batch 13/938, Loss: 2.3143510818481445
Epoch 1/5, Batch 14/938, Loss: 2.302727699279785
Epoch 1/5, Batch 15/938, Loss: 2.329989194869995
Epoch 1/5, Batch 16/938, Loss: 2.3001503944396973
Epoch 1/5, Batch 17/938, Loss: 2.301856279373169
Epoch 1/5, Batch 18/938, Loss: 2.304658889770508
Epoch 1/5, Batch 19/938, Loss: 2.3139379024505615
Epoch 1/5, Batch 20/938, Loss: 2.300968885421753
Epoch 1/5, Batch 21/

d:\miniconda3\envs\deephessian\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Error detected in MmBackward0. Traceback of forward call that caused the error:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\miniconda3\envs\deephessian\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\miniconda3\envs\deephessian\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\miniconda3\envs\deephessian\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "d:\miniconda3\envs\deephessian\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "d:\miniconda3\envs\deephessian\Lib\asyncio\base_events.py", line 640, in run_forever
    self._run_once()
  File "d:\miniconda3\envs\deephessian\Lib\asyncio\base_events.py", line 

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.FloatTensor [10, 100]], which is output 0 of AsStridedBackward0, is at version 4691; expected version 4690 instead. Hint: the backtrace further above shows the operation that failed to compute its gradient. The variable in question was changed in there or anywhere later. Good luck!